# AI Human — entrenar en Google Colab (GPU)

Humanoide RL estilo Toribash (MuJoCo **MJX** + **Brax PPO** sobre JAX/GPU). Este notebook
clona el repo, instala las deps de GPU y entrena. La **visualización** (Electron/Three.js) NO
corre acá: es de escritorio. Acá **solo se entrena**; después te bajás el `.params` y lo
visualizás local en Windows.

**Antes de empezar:** `Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU`
(T4 gratis, o A100 con Pro+).

Flujo: **1** GPU → **2** clonar → **3** instalar → **4** (opcional) Drive → **5** modo (de 0 /
continuar) → **6** hiperparámetros → **7** entrenar → **8** descargar.

## 1. Verificar GPU

In [ ]:
!nvidia-smi

## 2. Clonar / actualizar el repo

In [ ]:
import os
%cd /content
REPO = "https://github.com/Zalotron/AI-Human.git"
if os.path.isdir("AI-Human"):
    !cd AI-Human && git pull -q
else:
    !git clone -q $REPO
%cd /content/AI-Human
!pwd && ls

## 3. Instalar dependencias (JAX GPU + MuJoCo MJX + Brax)

Mismo set que en WSL2 (sin pinear versiones, dejando que pip resuelva). Si `Devices` de abajo
muestra **solo CPU**: `Entorno de ejecución → Cambiar tipo → GPU` y **reiniciar el entorno**
(`Entorno de ejecución → Reiniciar`), después volvé a correr desde la celda 2.

In [ ]:
!pip install -q -U "jax[cuda12]" mujoco mujoco-mjx brax numpy
import jax
print("JAX:", jax.__version__)
print("Devices:", jax.devices())   # tiene que aparecer un CudaDevice

## 4. (Opcional) Montar Google Drive para persistir entre sesiones

Colab **te desconecta** por inactividad (~90 min) o corta a las ~12 h → perderías el progreso.
Si montás Drive, el checkpoint (`mjx_policy.params`) y la cache de compilación XLA se guardan ahí:
si te desconectan, re-corrés el notebook y el training **reanuda solo** desde el último checkpoint.

Si NO corrés esta celda, todo queda en el disco efímero de Colab (se pierde al desconectar).

In [ ]:
#@title Montar Drive { display-mode: "form" }
from google.colab import drive
drive.mount("/content/drive")
import os
DRIVE_DIR = "/content/drive/MyDrive/AI-Human"  #@param {type:"string"}
os.makedirs(DRIVE_DIR, exist_ok=True)
os.environ["MJX_SAVE_PATH"] = os.path.join(DRIVE_DIR, "mjx_policy.params")
os.environ["JAX_COMPILATION_CACHE_DIR"] = os.path.join(DRIVE_DIR, ".jax_cache")
print("Checkpoint ->", os.environ["MJX_SAVE_PATH"])
print("Cache XLA  ->", os.environ["JAX_COMPILATION_CACHE_DIR"])

## 5. Modo: arrancar de 0 **o** continuar desde un `.params`

- **arrancar de 0**: borra el checkpoint y la cache → entrena desde cero.
- **continuar desde .params**: al ejecutar la celda aparece un **botón para subir** tu archivo
  `.params`; el training reanuda desde ahí (warm-start del actor + normalizador).

> Si el `.params` que subís tiene una obs distinta a la actual, el training lo **ignora y arranca
> de 0** (guarda anti-mismatch) — no crashea.

In [ ]:
#@title Elegí el modo y ejecutá { display-mode: "form" }
MODO = "arrancar de 0"  #@param ["arrancar de 0", "continuar desde .params"]

import os, shutil
from google.colab import files
CKPT = os.environ.get("MJX_SAVE_PATH", "mjx/mjx_policy.params")

if MODO == "arrancar de 0":
    for p in [CKPT, "mjx/.jax_cache", os.environ.get("JAX_COMPILATION_CACHE_DIR", "")]:
        if p and os.path.isdir(p):
            shutil.rmtree(p, ignore_errors=True)
        elif p and os.path.exists(p):
            os.remove(p)
    print("OK - reset: borrado checkpoint + cache XLA. El training arranca de 0.")
else:
    print("Subí tu archivo .params con el botón de abajo:")
    up = files.upload()
    if up:
        name = list(up.keys())[0]
        os.makedirs(os.path.dirname(CKPT) or ".", exist_ok=True)
        with open(CKPT, "wb") as f:
            f.write(up[name])
        print(f"OK - {name} -> {CKPT} ({len(up[name])/1e6:.1f} MB). El training reanuda desde acá.")
    else:
        print("No subiste nada. Volvé a ejecutar la celda, o cambiá MODO a 'arrancar de 0'.")

## 6. Hiperparámetros del run

Defaults pensados para **T4 (16 GB)**. `NUM_ENVS` debe cumplir `24576 % NUM_ENVS == 0`
(256 / 384 / 512 / 768 / 1024…). Si te da **OOM**, bajalo (256 → 128). `NUM_TIMESTEPS` completo
del proyecto es 200M (no entra en una sesión gratis); subilo si usás A100 / persistís en Drive.
El resto de la config (reward, física, self-collision) sale de `settings.json` del repo.

In [ ]:
#@title Configurar y ejecutar { display-mode: "form" }
NUM_ENVS = 256           #@param {type:"integer"}
NUM_TIMESTEPS = 50000000 #@param {type:"integer"}
EPISODE_LEN = 2000       #@param {type:"integer"}
THROW_EVERY = 100        #@param {type:"integer"}

import os
os.environ["MJX_NUM_ENVS"]      = str(NUM_ENVS)
os.environ["MJX_NUM_TIMESTEPS"] = str(NUM_TIMESTEPS)
os.environ["TRAIN_EPISODE_LEN"] = str(EPISODE_LEN)
os.environ["TRAIN_THROW_EVERY"] = str(THROW_EVERY)
assert 24576 % NUM_ENVS == 0, "NUM_ENVS debe dividir a 24576 (256/384/512/768/1024...)"
print("Config lista:", {k: os.environ[k] for k in
      ["MJX_NUM_ENVS","MJX_NUM_TIMESTEPS","TRAIN_EPISODE_LEN","TRAIN_THROW_EVERY"]})
print("Checkpoint:", os.environ.get("MJX_SAVE_PATH", "mjx/mjx_policy.params (efímero)"))

## 7. Entrenar

La **1ª vez compila** el grafo XLA (~1-3 min sin imprimir nada). Después imprime una línea por
update/eval: `step … | ep_len … | ep_reward … | parado …% | pose …% | ent … | st/s`.
Señales sanas: `parado`/`pose` subiendo, `ent ≈ 25`, `ep_len` cerca del `EPISODE_LEN`.
El checkpoint se guarda **en cada eval** (no solo al final).

In [ ]:
!cd mjx && python -u train_mjx.py

## 8. Descargar el checkpoint entrenado

Bajá el `.params` para usarlo local en la viz de Windows (`mjx/mjx_policy.params`), o para volver
a subirlo acá y **continuar** el entrenamiento (celda 5). Si montaste Drive, ya está guardado ahí.

In [ ]:
import os
from google.colab import files
CKPT = os.environ.get("MJX_SAVE_PATH", "mjx/mjx_policy.params")
if os.path.exists(CKPT):
    print("Descargando", CKPT, f"({os.path.getsize(CKPT)/1e6:.1f} MB)")
    files.download(CKPT)
else:
    print("No existe todavía:", CKPT, "- entrená primero (celda 7).")